In [31]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
from sklearn.datasets import make_blobs, load_iris # เรียกใช้ข้อมูลที่มากับ SciKit Learn
import pandas as pd

# คลีน Data และตรวจสอบหาค่าว่าง, Outlinder, ค่าซ้ำและอื่นๆ

In [32]:
restaurant = pd.read_csv("KFC.csv")
restaurant

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
0,10452,07-11-2022,Fries,3.49,573.07,Online,Gift Card,Tom Jackson,London
1,10453,07-11-2022,Beverages,2.95,745.76,Online,Gift Card,Pablo Perez,Madrid
2,10454,07-11-2022,Sides & Other,4.99,200.40,In-store,Gift Card,Joao Silva,Lisbon
3,10455,08-11-2022,Burgers,12.99,569.67,In-store,Credit Card,Walter Muller,Berlin
4,10456,08-11-2022,Chicken Sandwiches,9.95,201.01,In-store,Credit Card,Walter Muller,Berlin
...,...,...,...,...,...,...,...,...,...
249,10709,28-12-2022,Sides & Other,4.99,200.40,Drive-thru,Gift Card,Walter Muller,Berlin
250,10710,29-12-2022,Burgers,12.99,754.43,Drive-thru,Gift Card,Walter Muller,Berlin
251,10711,29-12-2022,Chicken Sandwiches,9.95,281.41,Drive-thru,Gift Card,Walter Muller,Berlin
252,10712,29-12-2022,Fries,3.49,630.37,Drive-thru,Gift Card,Walter Muller,Berlin


In [33]:
#ข้อมูลใน csv
restaurant.info()                                      # to show some basic informations about the dataset

<class 'pandas.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        254 non-null    int64  
 1   Date            254 non-null    str    
 2   Product         254 non-null    str    
 3   Price           254 non-null    float64
 4   Quantity        254 non-null    float64
 5   Purchase Type   254 non-null    str    
 6   Payment Method  254 non-null    str    
 7   Manager         254 non-null    str    
 8   City            254 non-null    str    
dtypes: float64(2), int64(1), str(6)
memory usage: 31.8 KB


In [34]:
#ตรวจสอบ missing values,หาค่าสถิติพื้นฐาน
restaurant[restaurant.isnull().any(axis=1)].head()
#restaurant.describe()

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City


In [35]:
#ตรวจหาข้อมูลซ้ำ
restaurant.duplicated().sum()

np.int64(0)

In [36]:
#check ชนิดข้อมูล
restaurant.dtypes

Order ID            int64
Date                  str
Product               str
Price             float64
Quantity          float64
Purchase Type         str
Payment Method        str
Manager               str
City                  str
dtype: object

In [37]:
#เช็คว่าควรเป็นตัวอักษร หรือตัวเลข
#restaurant["Product"] = pd.to_numeric(restaurant["Product"], errors="coerce")
#restaurant["Price"] = pd.to_numeric(restaurant["Price"], errors="coerce")
#restaurant["Quantity"] = pd.to_numeric(restaurant["Quantity"], errors="coerce")
#restaurant["Purchase Type"] = pd.to_numeric(restaurant["Purchase Type"], errors="coerce")
#restaurant["Payment Method"] = pd.to_numeric(restaurant["Payment Method"], errors="coerce")
#restaurant["Manager"] = pd.to_numeric(restaurant["Manager"], errors="coerce")
#restaurant["City"] = pd.to_numeric(restaurant["City"], errors="coerce")

In [38]:
Q1 = restaurant["Price"].quantile(0.25)
Q3 = restaurant["Price"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = restaurant[
    (restaurant["Price"] < lower) |
    (restaurant["Price"] > upper)
]

In [39]:
outliers

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
28,10482,13-11-2022,Fries,25.50,630.37,In-store,Credit Card,Joao Silva,Lisbon
29,10486,14-11-2022,Chicken Sandwiches,29.05,201.01,In-store,Credit Card,Joao Silva,Lisbon


In [40]:
#ปรับค่าให้เขียนตรงกันเช็คค่าที่ไม่ซ้ำกัน
print(restaurant["Product"].unique())
print(restaurant["Price"].unique())
print(restaurant["Quantity"].unique())
print(restaurant["Purchase Type"].unique())
print(restaurant["Payment Method"].unique())
print(restaurant["Manager"].unique())
print(restaurant["City"].unique())

<ArrowStringArray>
['Fries', 'Beverages', 'Sides & Other', 'Burgers', 'Chicken Sandwiches']
Length: 5, dtype: str
[ 3.49  2.95  4.99 12.99  9.95 25.5  29.05]
[573.07 745.76 200.4  569.67 201.01 554.27 677.97 630.37 523.48 508.08
 538.88 687.68 477.29 492.69 461.89 446.5  585.07 221.11 600.46 631.25
 646.65 677.44 241.21 261.31 692.84 281.41 723.63 301.51 754.43]
<ArrowStringArray>
['Online ', 'In-store ', 'Drive-thru ']
Length: 3, dtype: str
<ArrowStringArray>
[' Gift Card', ' Credit Card', ' Cash']
Length: 3, dtype: str
<ArrowStringArray>
[  'Tom      Jackson', '       Pablo Perez',      'Joao    Silva',
      'Walter Muller',      'Remy    Monet',         'Remy Monet',
  '       Remy Monet',     'Remy     Monet',        'Pablo Perez',
      'Pablo   Perez',       'Pablo  Perez',     'Pablo    Perez',
         'Joao Silva',        'Tom Jackson']
Length: 14, dtype: str
<ArrowStringArray>
['London', 'Madrid', 'Lisbon', 'Berlin', 'Paris']
Length: 5, dtype: str


In [41]:
#ลบช่องว่างส่วนเกิน

restaurant["Product"] = restaurant["Product"].str.strip()
restaurant["Purchase Type"] = restaurant["Purchase Type"].str.strip()
restaurant["Payment Method"] = restaurant["Payment Method"].str.strip()
restaurant["Manager"] = restaurant["Manager"].str.strip()
restaurant["City"] = restaurant["City"].str.strip()

In [42]:
print(restaurant["Product"].value_counts())
print(restaurant["Price"].value_counts())
print(restaurant["Quantity"].value_counts())
print(restaurant["Purchase Type"].value_counts())
print(restaurant["Payment Method"].value_counts())
print(restaurant["Manager"].value_counts())
print(restaurant["City"].value_counts())

Product
Burgers               52
Chicken Sandwiches    52
Fries                 51
Beverages             50
Sides & Other         49
Name: count, dtype: int64
Price
12.99    52
9.95     51
3.49     50
2.95     50
4.99     49
25.50     1
29.05     1
Name: count, dtype: int64
Quantity
200.40    49
201.01    36
630.37    35
677.97    34
745.76    16
573.07     9
221.11     8
687.68     7
569.67     6
523.48     6
554.27     5
538.88     5
477.29     5
508.08     4
677.44     4
492.69     3
461.89     3
241.21     3
281.41     3
585.07     2
646.65     2
692.84     2
446.50     1
600.46     1
631.25     1
261.31     1
723.63     1
301.51     1
754.43     1
Name: count, dtype: int64
Purchase Type
Online        107
In-store       86
Drive-thru     61
Name: count, dtype: int64
Payment Method
Credit Card    120
Cash            76
Gift Card       58
Name: count, dtype: int64
Manager
Tom Jackson         74
Joao Silva          70
Pablo Perez         43
Walter Muller       30
Remy Monet          2

# สร้าง feature ใหม่ และการ Encode

In [43]:
# แปลงข้อมูลวันที่ให้แตกออกมาเป็นข้อมูลย่อยๆ (วัน เดือน ปี) เพื่อให้สามารถพยากรได้อย่างแม่นยำ เพราะ ML ไม่รู้หรอกว่าอันไหนวันเดือนหรือปี
restaurant['Date'] = pd.to_datetime(restaurant['Date'], format='%d-%m-%Y')

# สร้างตีวแปรใหม่สำหรับวัน เหือน ปี แยกมาเลย เพื่อกัน data leak
restaurant['Year'] = restaurant['Date'].dt.year
restaurant['Month'] = restaurant['Date'].dt.month
restaurant['Day'] = restaurant['Date'].dt.day
restaurant['DayOfWeek'] = restaurant['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
restaurant['WeekOfYear'] = restaurant['Date'].dt.isocalendar().week

# คำนวณยอดค่าใช้จ่าย
restaurant['Total_Revenue'] = restaurant['Price'] * restaurant['Quantity']

print("=== DATE RANGE ===")
print(f"From: {restaurant['Date'].min()} to {restaurant['Date'].max()}")
print(f"Total days: {(restaurant['Date'].max() - restaurant['Date'].min()).days + 1}")

=== DATE RANGE ===
From: 2022-11-07 00:00:00 to 2022-12-29 00:00:00
Total days: 53


In [44]:
restaurant

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City,Year,Month,Day,DayOfWeek,WeekOfYear,Total_Revenue
0,10452,2022-11-07,Fries,3.49,573.07,Online,Gift Card,Tom Jackson,London,2022,11,7,0,45,2000.0143
1,10453,2022-11-07,Beverages,2.95,745.76,Online,Gift Card,Pablo Perez,Madrid,2022,11,7,0,45,2199.9920
2,10454,2022-11-07,Sides & Other,4.99,200.40,In-store,Gift Card,Joao Silva,Lisbon,2022,11,7,0,45,999.9960
3,10455,2022-11-08,Burgers,12.99,569.67,In-store,Credit Card,Walter Muller,Berlin,2022,11,8,1,45,7400.0133
4,10456,2022-11-08,Chicken Sandwiches,9.95,201.01,In-store,Credit Card,Walter Muller,Berlin,2022,11,8,1,45,2000.0495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,10709,2022-12-28,Sides & Other,4.99,200.40,Drive-thru,Gift Card,Walter Muller,Berlin,2022,12,28,2,52,999.9960
250,10710,2022-12-29,Burgers,12.99,754.43,Drive-thru,Gift Card,Walter Muller,Berlin,2022,12,29,3,52,9800.0457
251,10711,2022-12-29,Chicken Sandwiches,9.95,281.41,Drive-thru,Gift Card,Walter Muller,Berlin,2022,12,29,3,52,2800.0295
252,10712,2022-12-29,Fries,3.49,630.37,Drive-thru,Gift Card,Walter Muller,Berlin,2022,12,29,3,52,2199.9913


ลองทำ encode 2 แบบ ได้แก่ แบบ LabelEncode กับ One Hot Encode

# LabelEncoder

Label Encoding คือการแปลง Category ให้เป็น ตัวเลขตัวเดียวต่อ Category

In [69]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import preprocessing

# เพื่อความเรียบร้อยและกันงง จึงสร้าง dataFrame ใหม่มาเก็บข้อมูลเดิมแต่เป็นแบบ LabelEncode
restaurant_LabelEncoded = restaurant.copy()

label_encoders = {}
categorical_columns = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']

# Encode แบบ LabelEncode
model_encoder = preprocessing.LabelEncoder()

restaurant_LabelEncoded['Product'] = model_encoder.fit_transform(restaurant_LabelEncoded['Product'])
restaurant_LabelEncoded['Purchase Type'] = model_encoder.fit_transform(restaurant_LabelEncoded['Purchase Type'])
restaurant_LabelEncoded['Manager'] = model_encoder.fit_transform(restaurant_LabelEncoded['Manager'])
restaurant_LabelEncoded['City'] = model_encoder.fit_transform(restaurant_LabelEncoded['City'])
restaurant_LabelEncoded['Payment Method'] = model_encoder.fit_transform(restaurant_LabelEncoded['Payment Method'])

features = ['Quantity', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear','Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']

X = restaurant_LabelEncoded[features]
y = restaurant_LabelEncoded['Price']

# แบ่งข้อมูลเเพื่อใช้ train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# การปรับขนาดหรือปรับช่วงค่าของข้อมูลทดสอบ (Test Set) ให้มีสเกลหรือขอบเขตเดียวกันกับข้อมูลที่จะไป train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== PREPROCESSING COMPLETE ===")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Feature columns: {features}")

=== PREPROCESSING COMPLETE ===
Training set shape: (203, 10)
Test set shape: (51, 10)
Feature columns: ['Quantity', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear', 'Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']


In [70]:
# หลังจากเพิ่มหัวข้อใหม่เพื่อเตรียมไปใช้ train
restaurant_LabelEncoded

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City,Year,Month,Day,DayOfWeek,WeekOfYear,Total_Revenue
0,10452,2022-11-07,3,3.49,573.07,2,2,9,2,2022,11,7,0,45,2000.0143
1,10453,2022-11-07,0,2.95,745.76,2,2,5,3,2022,11,7,0,45,2199.9920
2,10454,2022-11-07,4,4.99,200.40,1,2,0,1,2022,11,7,0,45,999.9960
3,10455,2022-11-08,1,12.99,569.67,1,1,11,0,2022,11,8,1,45,7400.0133
4,10456,2022-11-08,2,9.95,201.01,1,1,11,0,2022,11,8,1,45,2000.0495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,10709,2022-12-28,4,4.99,200.40,0,2,11,0,2022,12,28,2,52,999.9960
250,10710,2022-12-29,1,12.99,754.43,0,2,11,0,2022,12,29,3,52,9800.0457
251,10711,2022-12-29,2,9.95,281.41,0,2,11,0,2022,12,29,3,52,2800.0295
252,10712,2022-12-29,3,3.49,630.37,0,2,11,0,2022,12,29,3,52,2199.9913


# One Hot Encode

คือวิธีแปลงข้อมูลประเภท Categorical หรือข้อมูลที่เป็นข้อความ/ประเภท ให้กลายเป็นตัวเลข เพื่อให้ Machine Learning สามารถนำไปใช้ได้

In [71]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

# เ# เพื่อความเรียบร้อยและกันงง จึงสร้าง dataFrame ใหม่มาเก็บข้อมูลเดิมแต่เป็นแบบ HotEncode
restaurant_OneHotEncoded = restaurant.copy()

categorical_columns = ['Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']

numeric_columns = ['Quantity', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear']

features = numeric_columns + categorical_columns

X = restaurant_OneHotEncoded[features]
y = restaurant_OneHotEncoded['Price']

# แบ่งข้อมูลเพื่อใช้ train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Encode แบบ One-Hot Encode โดยแบ่งช่วงเลขให้อยู่ใกล้เคียงกันโดยการ Scale
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ]
)

# Fit และ Transform ข้อมูล Train
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform ข้อมูล Test โดยใช้ encoder และ scaler เดียวกับ Train
X_test_scaled = preprocessor.transform(X_test)

print("=== PREPROCESSING COMPLETE ===")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

print("Processed training shape:", X_train_scaled.shape)
print("Processed test shape:", X_test_scaled.shape)

print(f"Feature columns: {features}")

=== PREPROCESSING COMPLETE ===
Training set shape: (203, 10)
Test set shape: (51, 10)
Processed training shape: (203, 31)
Processed test shape: (51, 31)
Feature columns: ['Quantity', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear', 'Product', 'Purchase Type', 'Payment Method', 'Manager', 'City']


In [79]:
# แสดงตาราง One-Hot Encoder
hot_encoded_data = preprocessor.named_transformers_['cat'].transform(
    X_train[categorical_columns]
)

hot_encoded_data = preprocessor.named_transformers_['cat'].get_feature_names_out(
    categorical_columns
)

hot_encoded_df = pd.DataFrame(
    encoded_data.toarray(),
    columns=encoded_columns,
    index=X_train.index
)

print(hot_encoded_df)

     Product_Beverages  Product_Burgers  Product_Chicken Sandwiches  \
38                 0.0              1.0                         0.0   
143                1.0              0.0                         0.0   
84                 0.0              0.0                         0.0   
55                 0.0              0.0                         0.0   
218                1.0              0.0                         0.0   
..                 ...              ...                         ...   
106                0.0              0.0                         1.0   
14                 0.0              0.0                         0.0   
92                 0.0              0.0                         0.0   
179                0.0              0.0                         0.0   
102                0.0              0.0                         0.0   

     Product_Fries  Product_Sides & Other  Purchase Type_Drive-thru  \
38             0.0                    0.0                       0.0   
143  

เลือกใช้ระหว่าง LabelEncode หรือ HotEncode ????